In [1]:
import os
from pathlib import Path

# 检查数据是否已存在
dataset_path = Path("Emotion-Detection-1")
data_yaml = dataset_path / "data.yaml"
dataset = None
dataset_location = None
MODEL_NAME = "yolo11s.pt"  # 模型文件名

if data_yaml.exists():
    print(f"[OK] 数据集已存在: {dataset_path.absolute()}")
    print("跳过下载，直接使用现有数据")
    dataset_location = str(dataset_path.absolute())
else:
    print("数据集不存在，开始下载...")
    !pip install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key="wAFpXIOS5YhwXaos60c3")
    project = rf.workspace("computer-vision-projects-zhogq").project("emotion-detection-y0svj")
    version = project.version(1)
    dataset = version.download("yolov11")
    dataset_location = dataset.location

print(f"数据集路径: {dataset_location}")


[OK] 数据集已存在: E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu\Emotion-Detection-1
跳过下载，直接使用现有数据
数据集路径: E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu\Emotion-Detection-1


In [2]:
dataset_location

'E:\\work\\PycharmProjects\\Multi-modal-Fall-Detection-and-Affective-Care\\qingxu\\Emotion-Detection-1'

In [ ]:
# 开始训练
!pip install ultralytics
from ultralytics import YOLO

print(f"开始训练: {MODEL_NAME}")
print(f"数据集: {dataset_location}")
print(f"训练轮数: 150 epochs (比原来的 70 增加)")
print(f"批次大小: 16")
print(f"图像尺寸: 640")

# 训练命令
!yolo task=detect mode=train epochs=150 batch=16 plots=True model='model/{MODEL_NAME}' data={dataset_location}/data.yaml imgsz=640 patience=50 cos_lr=True device=0 workers=8


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
开始训练: yolo11s.pt
数据集: E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu\Emotion-Detection-1
训练轮数: 150 epochs (比原来的 70 增加)
批次大小: 16
图像尺寸: 640



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 切换到 notebook 所在目录（原为 Google Colab 的 Drive 挂载路径，本地 Windows 环境不存在，已改为可移植写法）
import os
from pathlib import Path

os.chdir(Path(dataset_location).parent)  # qingxu 目录
print(f"当前工作目录: {os.getcwd()}")

当前工作目录: E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu


In [4]:
ls

 ������ E �еľ��� ����
 �������к��� 8842-39E7

 E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu ��Ŀ¼

2026/11/18  01:00    <DIR>          .
2026/11/18  00:57    <DIR>          ..
2026/11/16  11:34    <DIR>          __pycache__
2026/11/16  11:31            31,294 app.py
2026/11/05  23:11        10,586,597 best.onnx.bak
2026/11/17  00:25             5,205 compare_models.py
2026/11/11  19:08               531 config.yaml
2026/11/05  23:11           124,954 confusion_matrix.png
2026/11/09  15:51    <DIR>          dataset
2026/11/09  16:07    <DIR>          Emotion-Detection-1
2026/11/18  01:00            26,964 Face_Emotion_Detection_Yolo_11.ipynb
2026/11/05  23:11           229,065 labels.jpg
2026/11/05  23:11             1,088 LICENSE
2026/11/17  00:31    <DIR>          model
2026/11/11  00:51    <DIR>          output
2026/11/17  01:36    <DIR>          pro_emotion
2026/11/16  15:42             9,164 README.md
2026/11/09  23:34    <DIR>          runs
2026/11/1

In [5]:
# 导出为 ONNX
from ultralytics import YOLO
import shutil
from pathlib import Path


# 加载训练好的模型
best_pt = "./runs/detect/train-3/weights/best.pt"
model = YOLO(best_pt)

# 导出为 ONNX
print("导出 ONNX 模型...")
onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
print(f"[OK] ONNX 模型已导出: {onnx_path}")

# 复制为带模型名称的文件
model_suffix = MODEL_NAME.replace('.pt', '')  # yolo11s
dest_name = f"best_{model_suffix}.onnx"
dest_path = Path(dataset_location).parent / dest_name
shutil.copy(onnx_path, dest_path)
print(f"[OK] 模型已复制为: {dest_path}")

print("下一步:")
print(f"1. 测试新模型: python app.py --model {dest_name}")
print(f"2. 如果效果满意，替换 best.onnx: copy {dest_name} best.onnx")


导出 ONNX 模型...
Ultralytics 8.4.208  Python-3.11.6 torch-2.6.0+cu124 CPU (12th Gen Intel Core i7-12700KF)
YOLO11s summary (fused): 100 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from 'runs\detect\train-3\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (18.3 MB)

ONNX: starting export with onnx 1.16.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success  4.8s, saved as 'runs\detect\train-3\weights\best.onnx' (36.2 MB)

Export complete (5.1s)
Results saved to E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu\runs\detect\train-3\weights\best.onnx
Predict:         yolo predict task=detect model=runs\detect\train-3\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs\detect\train-3\weights\best.onnx imgsz=640 data=E:\work\PycharmProjects\Multi-modal-Fall-Detection-and-Affective-Care\qingxu\Emotion-Detection-1\data.yaml 
Visualize:       https://netron.